# Practice houses — synthetic prices

This notebook uses **made-up** listing data (`housing-sample.csv`) so you can repeat the same workflow as the insurance project without worrying about real addresses or PII. The goal is still practice: explore the table, clean encodings, add one engineered bucket, run correlation and chi-square screens, scale a few numerics, then freeze a **`final_df`** for modeling later.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("housing-sample.csv")

In [ ]:
df

## First look at the data

Check size, a few rows, dtypes, missing values, and quick numeric summaries before changing anything.

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df["neighborhood"].value_counts()

In [ ]:
df["garage"].value_counts()

## Plots that usually help

Histograms and box plots show skew and outliers; a correlation heatmap is only for numeric columns and only captures *linear* pairwise patterns.

In [ ]:
numeric_cols = ["sqft_living", "bedrooms", "age_years", "price"]
for col in numeric_cols:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[col], kde=True)
    plt.title(col)
    plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=df["price"])
plt.title("price")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f")
plt.show()

## Cleaning and simple encodings

Work on a copy, drop duplicate rows if any, turn `garage` into a binary flag, then expand `neighborhood` with dummies. With `drop_first=True`, pandas drops one baseline category (here **`east`**, because the dummy names are sorted alphabetically—always glance at `df_cleaned.columns` after `get_dummies` so your chi-square list matches reality).

In [ ]:
df_cleaned = df.copy()

In [ ]:
df_cleaned.drop_duplicates(inplace=True)
df_cleaned.shape

In [ ]:
df_cleaned["garage"] = df_cleaned["garage"].map({"no": 0, "yes": 1})
df_cleaned.rename(columns={"garage": "has_garage"}, inplace=True)
df_cleaned.head()

In [ ]:
df_cleaned = pd.get_dummies(
    df_cleaned, columns=["neighborhood"], drop_first=True
)
df_cleaned.head()

In [ ]:
# bool/int consistency for modeling
df_cleaned = df_cleaned.astype(
    {c: int for c in df_cleaned.select_dtypes(include=["bool"]).columns}
)
df_cleaned.dtypes

## Bucketing house age

`pd.cut` turns `age_years` into ordered bands (new vs mature vs older). That can capture non-linear effects compared to using age only as a straight line. Then dummy-encode the bands like we did for region.

In [ ]:
df_cleaned["age_bracket"] = pd.cut(
    df_cleaned["age_years"],
    bins=[-1, 15, 40, 100],
    labels=["new", "mature", "older"],
)
df_cleaned["age_bracket"].value_counts()

In [ ]:
df_cleaned = pd.get_dummies(
    df_cleaned, columns=["age_bracket"], drop_first=True
)
df_cleaned = df_cleaned.astype(
    {c: int for c in df_cleaned.select_dtypes(include=["bool"]).columns}
)
df_cleaned.head()

## Standardizing a few numerics

`StandardScaler` puts `sqft_living`, `bedrooms`, and `age_years` on a comparable scale. Dummy columns stay 0/1. For real projects, fit the scaler on **train** only; here we keep the notebook short and fit on the full table like the first project.

In [ ]:
from sklearn.preprocessing import StandardScaler

scale_cols = ["sqft_living", "bedrooms", "age_years"]
scaler = StandardScaler()
df_cleaned[scale_cols] = scaler.fit_transform(df_cleaned[scale_cols])
df_cleaned.head()

## Linear association with `price` (Pearson)

Pearson *r* measures straight-line relationships. Sorting helps you see which engineered columns move most *linearly* with price in this sample. Correlation is not causation, and including `price` in the feature list would trivially correlate with itself.

In [ ]:
from scipy.stats import pearsonr

numeric_features = [
    c
    for c in df_cleaned.columns
    if c != "price" and df_cleaned[c].dtype in ("int64", "float64", "int32", "float32")
]
correlation = {
    feat: pearsonr(df_cleaned[feat], df_cleaned["price"])[0]
    for feat in numeric_features
}
corr_df = pd.DataFrame(
    list(correlation.items()), columns=["feature", "pearson_r"]
).sort_values("pearson_r", key=lambda s: s.abs(), ascending=False)
corr_df

## Chi-square: categories vs price quartiles

Turn `price` into four frequency-based buckets with `pd.qcut`, cross-tab each categorical column with those buckets, and run `chi2_contingency`. Small p-values suggest association worth noticing for feature screening (not final proof).

In [ ]:
categorical_features = [
    "has_garage",
    "neighborhood_north",
    "neighborhood_south",
    "age_bracket_mature",
    "age_bracket_older",
]

In [ ]:
from scipy.stats import chi2_contingency

alpha = 0.05
df_cleaned["price_bin"] = pd.qcut(df_cleaned["price"], q=4, labels=False)
chi2_rows = []
for col in categorical_features:
    table = pd.crosstab(df_cleaned[col], df_cleaned["price_bin"])
    chi2_stat, p_value, _, _ = chi2_contingency(table)
    chi2_rows.append(
        {
            "feature": col,
            "chi2_stat": chi2_stat,
            "p_value": p_value,
            "decision": (
                "reject_null_keep"
                if p_value < alpha
                else "fail_to_reject_drop_candidate"
            ),
        }
    )
chi2_df = pd.DataFrame(chi2_rows).sort_values("p_value")
chi2_df

## Columns to carry into the next step

`final_df` is the modeling slice: scaled numerics, dummies, and the target `price`. `price_bin` was only for the chi-square screen — drop it from the modeling frame unless you intentionally model buckets.

In [ ]:
final_df = df_cleaned.drop(columns=["price_bin"], errors="ignore")[
    [
        "sqft_living",
        "bedrooms",
        "age_years",
        "has_garage",
        "neighborhood_north",
        "neighborhood_south",
        "age_bracket_mature",
        "age_bracket_older",
        "price",
    ]
]
final_df.head()